# BLAST

![BLAST](https://proto-bio.github.io/proto-assets/images/tool/blast/hero.png)

This notebook demonstrates the two BLAST tools: `run_create_blast_db` builds a local BLAST database from a FASTA file, and `run_blast_search` searches sequences against either a local database or NCBI's online servers. BLAST+ binaries are downloaded automatically on first use, so no manual installation is required.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("blast")
display_overview("blast")
display_docs_section("blast", "Background")

## Available tools

In [ ]:
display_available_tools("blast")

### `run_create_blast_db`

`run_create_blast_db` builds a local BLAST database from a FASTA file using the `makeblastdb` utility from BLAST+. It produces a set of indexed database files whose base path can then be passed directly to `BlastSearchConfig.local_db`. Both nucleotide and protein databases are supported via the `dbtype` parameter.

In [ ]:
import tempfile
from pathlib import Path

from proto_tools import (
    CreateBlastDbInput,
    CreateBlastDbConfig,
    run_create_blast_db,
)

In [ ]:
# Display input docs
display_api_reference("blast", "input", "run_create_blast_db")

# Write a small FASTA with three well-characterized human globin proteins.
# The handle keeps the inputs and generated database alive across cells; the
# final cell cleans it up so nothing is left on disk.
tmp_dir_handle = tempfile.TemporaryDirectory(prefix="blast_example_")
tmp_dir = Path(tmp_dir_handle.name)

fasta_path = Path(tmp_dir) / "test_sequences.fasta"
fasta_path.write_text(
    ">sp|P69905|HBA_HUMAN Hemoglobin subunit alpha\n"
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH\n"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL\n"
    "LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR\n"
    ">sp|P68871|HBB_HUMAN Hemoglobin subunit beta\n"
    "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST\n"
    "PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDP\n"
    "ENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH\n"
    ">sp|P02144|MYG_HUMAN Myoglobin\n"
    "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL\n"
    "KSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIP\n"
    "VKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG\n"
)
print(f"Wrote test FASTA to: {fasta_path}")

db_input = CreateBlastDbInput(fasta=str(fasta_path))

In [ ]:
# Display config docs
display_api_reference("blast", "config", "run_create_blast_db")

# Create a protein database with a descriptive title | see docs above for all fields
db_config = CreateBlastDbConfig(dbtype="prot", title="Test Protein DB")

In [ ]:
# Run database creation
db_result = run_create_blast_db(db_input, db_config)

In [ ]:
# Display output docs
display_api_reference("blast", "output", "run_create_blast_db")

# The db_path is passed directly to BlastSearchConfig.local_db for searches
print(f"Database created at: {db_result.db_path}")

### `run_blast_search`

`run_blast_search` performs sequence similarity search in either local or online mode. Local mode runs BLAST+ against a database created by `run_create_blast_db`, while online mode submits the query to NCBI's QBLAST service and returns hits from large public databases such as SwissProt or nr. The query accepts either a raw sequence string or a path to a FASTA file, and results are returned as a list of `BlastHit` objects carrying the standard 12 tabular fields plus the aligned query, aligned subject, and BTOP traceback when the backend returns them.

In [ ]:
from proto_tools import (
    BlastSearchInput,
    BlastSearchConfig,
    run_blast_search,
)

In [ ]:
# Display input docs
display_api_reference("blast", "input", "run_blast_search")

# Write a FASTA query file with the hemoglobin beta sequence
query_seq = "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH"

query_path = Path(tmp_dir) / "query.fasta"
query_path.write_text(f">query_hbb\n{query_seq}\n")

search_input = BlastSearchInput(query=str(query_path))

In [ ]:
# Display config docs
display_api_reference("blast", "config", "run_blast_search")

# Local blastp search against the database we just built | see docs above for all fields
search_config = BlastSearchConfig(
    search_mode="local",
    program="blastp",
    local_db=db_result.db_path,
    num_threads=2,
)

In [ ]:
# Run the local BLAST search
result = run_blast_search(search_input, search_config)

In [ ]:
# Display output docs
display_api_reference("blast", "output", "run_blast_search")

# HBB should self-hit at 100% identity; HBA should appear at ~43% identity
print(f"Found {result.num_hits} hits\n")
for hit in result.hits:
    print(f"  {hit.sseqid}: {hit.pident:.1f}% identity, e-value={hit.evalue:.1e}, bitscore={hit.bitscore:.1f}")

#### Online search

For searches against large public databases, set `search_mode="online"`. The query is submitted to NCBI's QBLAST service and requires an internet connection. The raw sequence string can be passed directly to `BlastSearchInput` without writing a FASTA file.

In [ ]:
# Online search accepts a raw sequence string directly
online_input = BlastSearchInput(query=query_seq)
online_config = BlastSearchConfig(
    search_mode="online",
    program="blastp",
    database="swissprot",
    hitlist_size=5,
)

online_result = run_blast_search(online_input, online_config)
print(f"Found {online_result.num_hits} hits\n")
for hit in online_result.hits:
    print(f"  {hit.sseqid}: {hit.pident:.1f}% identity, e-value={hit.evalue:.1e}, bitscore={hit.bitscore:.1f}")

## Export Results

Search results can be exported to CSV or JSON for downstream analysis. CSV produces one row per hit with the standard 12 tabular fields; JSON additionally carries the alignment-detail fields (`qseq`, `sseq`, `btop`).

In [ ]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="blast_search_local_results", export_path=output_dir, file_format="csv")
online_result.export(name="blast_search_online_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

In [ ]:
# Clean up the temporary directory holding the example inputs and database
tmp_dir_handle.cleanup()